In [ ]:
import textwrap
from collections import Counter

import altair as alt
import pandas as pd

In [ ]:
def format_percentage(row, col_name):
    count = int(row["counts"])
    percentage = row["percentage"] * 100
    w = "response"
    if count > 1:
        w += "s"
    return f"{percentage:.0f}% ({count} {w})"

In [ ]:
def add_formatted_column(data):
    total = data["counts"].sum()
    data["percentage"] = data["counts"] / total
    data["formatted"] = data.apply(format_percentage, axis=1, col_name="Time")
    return data

In [ ]:
def wrap(text, width):
    return textwrap.wrap(text, width)

In [ ]:
def alt_text(title, options, counts, order):
    text = []
    text.append(f"{title}")
    if order == "-x":
        pairs = zip(options, counts)
        sorted_pairs = sorted(pairs, key=lambda x: x[1], reverse=True)
        for o, c in sorted_pairs:
            text.append(f"- {o}: {c}")
    else:
        d = {o: c for o, c in zip(options, counts)}
        for o in order:
            if o in d:
                text.append(f"- {o}: {d[o]}")
    print("\n".join(text))
    with open("alt-text.md", "a") as f:
        f.write("\n\n")
        f.write("\n".join(text))

In [ ]:
def plot(data, title, order, output_file, subtitle=[]):
    c = Counter(data[question].dropna().tolist())
    options, counts = zip(*c.items())

    alt_text(title, options, counts, order)

    data_question = pd.DataFrame(
        {
            "options": options,
            "counts": counts,
        }
    )

    _data = add_formatted_column(data_question)

    bars = (
        alt.Chart(
            _data,
            title=alt.Title(
                wrap(title, width=40),
                subtitle=subtitle,
            ),
        )
        .mark_bar()
        .encode(
            x=alt.X("percentage:Q", axis=alt.Axis(format=".0%", title=None)),
            y=alt.Y("options:N", axis=alt.Axis(title=None, labels=True), sort=order),
            color=alt.Color("options:N", legend=None),
        )
    )

    text = (
        alt.Chart(_data)
        .mark_text(align="left", baseline="middle", dx=3)
        .encode(
            x=alt.X("percentage:Q"),
            y=alt.Y("options:N", sort=order),
            text=alt.Text("formatted"),
        )
    )

    chart = bars + text

    # chart = chart.properties(width=600, height=250)

    chart = chart.configure_title(
        fontSize=15,
        subtitleFontSize=15,
        anchor="start",
        orient="bottom",
        offset=15,
        subtitleColor="gray",
    )

    chart.show()
    chart.save(output_file, scale_factor=2)

In [ ]:
data = pd.read_csv("data.txt", sep=";")

In [ ]:
question = "In your estimate, how much time per month have you saved as a result of attending a CodeRefinery workshop?"
order = ["No time saved", "Minutes", "Hours", "Days"]

plot(data, question, order, "time-saved.png")

In [ ]:
question = "After attending the workshop, would you judge your code to be more reusable or not more reusable?"
order = ["My code is more reusable", "My code is not more reusable", "Not sure"]

plot(data, question, order, "reusable.png")

In [ ]:
question = "After attending the workshop, has it become easier or not for you to collaborate on software development with your colleagues and collaborators?"
order = ["Collaboration is easier", "Collaboration is not easier", "Not sure"]

plot(data, question, order, "collaboration.png")

In [ ]:
question = "Have you introduced one or more of your colleagues to new tools or practices as a result of the workshop?"
order = [
    "I have introduced one or more of my colleagues to new tools or practices",
    "I have not introduced one or more of my colleagues to new tools or practices",
    "Not sure",
]

plot(data, question, order, "colleagues.png")

In [ ]:
question = "How likely is it that you would recommend CodeRefinery workshop to a friend or colleague?"
order = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]

plot(
    data,
    question,
    order,
    "recommending.png",
    subtitle="0 means definitely not. 10 means definitely yes.",
)

In [ ]:
question = "What would be your preferred delivery style for this workshop?"
order = [
    "MOOC (Asynchronous learning on learning platform)",
    "Prerecorded lectures and live discussion sessions",
    "Live taught online lectures with exercises",
    "Live taught online lectures with demos",
    "In-person teaching with exercises",
    "In-person teaching with demos",
    "Local classroom for streamed workshop with exercises"
]

plot(data, question, order, "pre-recorded-or-live-or-in-person.png")

In [ ]:
question = "Participation style"
order = [
    "Individual learner",
    "Learner in a team (with your colleagues/friends)",
    "Learner in a local classroom",
    "Other",
]

plot(data, question, order, "participation-style.png")

In [ ]:
question = "Career stage"
order = [
    "Undergraduate student",
    "Graduate student/ PhD student",
    "Postdoc",
    "Researcher",
    "Professor",
    "Research software engineer/ Scientific programmer",
    "Industry",
    "Other",
]

plot(data, question, order, "career-stage.png")

In [ ]:
question = "Academic discipline"
order = "-x"

plot(data, question, order, "academic-discipline.png")